<a href="https://colab.research.google.com/github/Myria255/BOOTCAMP-TTA/blob/main/W7D1_Exercises_XP_NLP_Preprocessing_NER_POS_Word2Vec.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 7 — Day 1: Intro to NLP — Exercises XP

**Developers Institute & Sira Labs**  
**COT GenAI & Machine Learning Bootcamp — 2026**

## Objectifs

Dans ce notebook, nous allons :

- nettoyer et prétraiter des avis textuels ;
- appliquer la reconnaissance d’entités nommées (**NER**) avec spaCy ;
- effectuer l’étiquetage grammatical (**POS tagging**) avec NLTK ;
- créer des représentations vectorielles avec **Word2Vec** ;
- visualiser les embeddings en deux dimensions avec **PCA** ;
- comparer les résultats obtenus sur les textes bruts et les textes prétraités.

> Exécutez les cellules dans l’ordre avec **Runtime → Run all** dans Google Colab.

In [ ]:
# Installation des bibliothèques nécessaires
# L'option -q réduit les messages affichés pendant l'installation.

!pip install -q pandas numpy matplotlib scikit-learn nltk spacy gensim
!python -m spacy download en_core_web_sm -q

In [ ]:
# Importation des bibliothèques et téléchargement des ressources NLTK

import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import nltk
import spacy

from IPython.display import display
from nltk import pos_tag, word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from gensim.models import Word2Vec
from sklearn.decomposition import PCA

warnings.filterwarnings("ignore")

# Ressources nécessaires au prétraitement et au POS tagging
resources = [
    "punkt",
    "punkt_tab",
    "stopwords",
    "wordnet",
    "omw-1.4",
    "averaged_perceptron_tagger",
    "averaged_perceptron_tagger_eng",
    "tagsets",
]

for resource in resources:
    try:
        nltk.download(resource, quiet=True)
    except Exception:
        pass

# Chargement du modèle anglais de spaCy
nlp = spacy.load("en_core_web_sm")

print("Toutes les bibliothèques et ressources sont prêtes.")

## Jeu de données

Nous conservons deux versions des données :

1. **raw_df** : les avis originaux ;
2. **cleaned_df** : les avis après prétraitement.

In [ ]:
# Création du jeu de données brut

data = {
    "Review": [
        "At McDonald's the food was ok and the service was bad.",
        "I would not recommend this Japanese restaurant to anyone.",
        "I loved this restaurant when I traveled to Thailand last summer.",
        "The menu of Loving has a wide variety of options.",
        "The staff was friendly and helpful at Google's employees restaurant.",
        "The ambiance at Bella Italia is amazing, and the pasta dishes are delicious.",
        "I had a terrible experience at Pizza Hut. The pizza was burnt, and the service was slow.",
        "The sushi at Sushi Express is always fresh and flavorful.",
        "The steakhouse on Main Street has a cozy atmosphere and excellent steaks.",
        "The dessert selection at Sweet Treats is to die for!"
    ]
}

raw_df = pd.DataFrame(data)

print(f"Nombre d'avis : {len(raw_df)}")
display(raw_df)

# Exercice 1 — Prétraitement, NER et POS tagging

## 1. Fonction `preprocess_text()`

La fonction réalise les opérations suivantes :

- conversion en minuscules ;
- tokenisation ;
- suppression de la ponctuation et des éléments non alphabétiques ;
- suppression des stopwords anglais ;
- lemmatisation ;
- reconstruction du texte nettoyé.

In [ ]:
# Initialisation des outils de prétraitement

english_stopwords = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()


def preprocess_text(text):
    '''
    Nettoie une chaîne de caractères et retourne une chaîne prétraitée.
    '''
    if not isinstance(text, str):
        return ""

    # Conversion en minuscules
    text = text.lower()

    # Tokenisation
    tokens = word_tokenize(text)

    # Suppression de la ponctuation et des tokens non alphabétiques
    tokens = [token for token in tokens if token.isalpha()]

    # Suppression des stopwords
    tokens = [token for token in tokens if token not in english_stopwords]

    # Lemmatisation
    lemmas = [lemmatizer.lemmatize(token) for token in tokens]

    return " ".join(lemmas)


# Vérification sur un premier exemple
sample_review = raw_df.loc[0, "Review"]

print("Texte brut :")
print(sample_review)
print("\nTexte prétraité :")
print(preprocess_text(sample_review))

In [ ]:
# 2. Application du prétraitement à tout le jeu de données

cleaned_df = raw_df.copy()
cleaned_df["Cleaned_Review"] = cleaned_df["Review"].apply(preprocess_text)

print("Jeu de données contenant les textes bruts et nettoyés :")
display(cleaned_df)

# Version ne contenant que le texte nettoyé
preprocessed_df = cleaned_df[["Cleaned_Review"]].copy()

print("\nJeu de données prétraité :")
display(preprocessed_df)

## 3. Fonction `perform_ner()`

La reconnaissance d’entités nommées identifie notamment :

- `ORG` : organisation ;
- `GPE` : pays, ville ou autre entité géopolitique ;
- `DATE` : date ou période ;
- `PERSON` : personne ;
- `FAC` : bâtiment, route ou autre installation.

In [ ]:
def perform_ner(text):
    '''
    Applique la reconnaissance d'entités nommées avec spaCy.

    Retour :
        liste de tuples (texte_entité, label)
    '''
    if not isinstance(text, str) or not text.strip():
        return []

    doc = nlp(text)
    return [(entity.text, entity.label_) for entity in doc.ents]


# Vérification de la fonction sur un exemple
ner_example = raw_df.loc[2, "Review"]

print("Texte :")
print(ner_example)
print("\nEntités détectées :")
print(perform_ner(ner_example))

## 4. Fonction `perform_pos_tagging()`

Le POS tagging associe à chaque mot sa catégorie grammaticale.  
Par exemple :

- `NN` : nom commun singulier ;
- `NNS` : nom commun pluriel ;
- `JJ` : adjectif ;
- `VB` : verbe à l’infinitif ;
- `VBD` : verbe au passé ;
- `RB` : adverbe ;
- `NNP` : nom propre singulier.

In [ ]:
def perform_pos_tagging(text):
    '''
    Tokenise le texte puis applique le POS tagging avec NLTK.

    Retour :
        liste de tuples (token, étiquette_POS)
    '''
    if not isinstance(text, str) or not text.strip():
        return []

    tokens = word_tokenize(text)
    return pos_tag(tokens)


# Vérification de la fonction sur un exemple
pos_example = raw_df.loc[5, "Review"]

print("Texte :")
print(pos_example)
print("\nPOS tags :")
print(perform_pos_tagging(pos_example))

In [ ]:
# 5. Application de NER et du POS tagging aux textes bruts et nettoyés

analysis_df = cleaned_df.copy()

analysis_df["NER_Raw"] = analysis_df["Review"].apply(perform_ner)
analysis_df["NER_Cleaned"] = analysis_df["Cleaned_Review"].apply(perform_ner)

analysis_df["POS_Raw"] = analysis_df["Review"].apply(perform_pos_tagging)
analysis_df["POS_Cleaned"] = analysis_df["Cleaned_Review"].apply(perform_pos_tagging)

pd.set_option("display.max_colwidth", None)

print("Comparaison des entités nommées :")
display(
    analysis_df[
        ["Review", "Cleaned_Review", "NER_Raw", "NER_Cleaned"]
    ]
)

print("\nComparaison des étiquettes grammaticales :")
display(
    analysis_df[
        ["Review", "Cleaned_Review", "POS_Raw", "POS_Cleaned"]
    ]
)

In [ ]:
# Affichage lisible, avis par avis

for index, row in analysis_df.iterrows():
    print("=" * 100)
    print(f"AVIS {index + 1}")
    print("-" * 100)
    print("Texte brut :", row["Review"])
    print("Texte nettoyé :", row["Cleaned_Review"])
    print("NER brut :", row["NER_Raw"])
    print("NER nettoyé :", row["NER_Cleaned"])
    print("POS brut :", row["POS_Raw"])
    print("POS nettoyé :", row["POS_Cleaned"])
    print()

### Analyse des résultats de l’exercice 1

Le prétraitement réduit fortement le bruit textuel : la ponctuation, les mots fréquents peu informatifs et certaines variations morphologiques sont supprimés. Cette transformation est utile pour plusieurs tâches statistiques ou pour entraîner Word2Vec sur un petit corpus.

Cependant, les résultats montrent aussi une perte d’information :

- la conversion en minuscules fait disparaître un indice important utilisé pour reconnaître les noms propres ;
- la suppression de mots peut casser le contexte d’une entité ;
- la NER est généralement plus fiable sur les phrases brutes, car spaCy utilise la structure, la casse et le contexte complet ;
- le POS tagging sur le texte brut donne une structure grammaticale plus réaliste ;
- le POS tagging du texte nettoyé porte surtout sur des mots informatifs isolés, sans représenter toute la phrase.

**Conclusion :** le niveau de prétraitement doit être adapté à la tâche. Pour la NER et l’analyse grammaticale, il est préférable de conserver le texte original. Pour Word2Vec, la classification ou l’analyse de fréquence, une version nettoyée peut être plus pertinente.

In [ ]:
# Aide facultative pour comprendre une étiquette POS

print("Signification de l'étiquette NN :\n")
nltk.help.upenn_tagset("NN")

# Exercice 2 — Création et visualisation des embeddings Word2Vec

## 1. Entraînement du modèle Word2Vec

Word2Vec attend une liste de phrases, chaque phrase étant représentée par une liste de tokens.

In [ ]:
# Transformation des avis nettoyés en listes de tokens

tokenized_reviews = [
    review.split()
    for review in cleaned_df["Cleaned_Review"]
    if isinstance(review, str) and review.strip()
]

print("Exemple de phrase tokenisée :")
print(tokenized_reviews[0])

# Création du modèle Word2Vec
word2vec_model = Word2Vec(
    sentences=tokenized_reviews,
    vector_size=50,  # nombre de dimensions de chaque vecteur
    window=3,        # nombre de mots observés à gauche et à droite
    min_count=1,     # conserve même les mots apparaissant une seule fois
    workers=1,       # résultat plus reproductible
    sg=1,            # Skip-gram
    epochs=200,      # davantage d'itérations pour ce petit corpus
    seed=42
)

vocabulary_size = len(word2vec_model.wv.index_to_key)
vector_size = word2vec_model.wv.vector_size
matrix_shape = word2vec_model.wv.vectors.shape

print("\nInformations sur le modèle Word2Vec")
print(f"Taille du vocabulaire : {vocabulary_size} mots")
print(f"Dimension d'un vecteur : {vector_size}")
print(f"Dimensions de la matrice complète : {matrix_shape}")

print(
    "\nInterprétation : chaque ligne correspond à un mot du vocabulaire "
    "et chaque colonne à une caractéristique numérique apprise par Word2Vec."
)

In [ ]:
# Consultation de quelques vecteurs et similarités

example_words = [
    word for word in ["restaurant", "service", "pizza", "sushi", "delicious"]
    if word in word2vec_model.wv
]

for word in example_words:
    print("=" * 80)
    print(f"Mot : {word}")
    print("Premières valeurs du vecteur :", word2vec_model.wv[word][:10])

    similar_words = word2vec_model.wv.most_similar(word, topn=5)
    print("Mots les plus similaires :", similar_words)

## 2. Fonction `plot_word_embeddings()`

Les vecteurs Word2Vec possèdent 50 dimensions et ne peuvent donc pas être représentés directement sur un graphique classique. Nous utilisons **PCA** pour les projeter en deux dimensions, puis nous plaçons chaque mot sur un nuage de points.

In [ ]:
def plot_word_embeddings(word2vec_object):
    '''
    Réduit les embeddings Word2Vec à deux dimensions avec PCA
    et les représente dans un scatter plot annoté.
    '''
    words = word2vec_object.wv.index_to_key
    vectors = word2vec_object.wv[words]

    # Réduction de 50 dimensions vers 2 dimensions
    pca = PCA(n_components=2)
    reduced_vectors = pca.fit_transform(vectors)

    plt.figure(figsize=(16, 12))
    plt.scatter(
        reduced_vectors[:, 0],
        reduced_vectors[:, 1],
        alpha=0.75
    )

    for index, word in enumerate(words):
        plt.annotate(
            word,
            xy=(reduced_vectors[index, 0], reduced_vectors[index, 1]),
            xytext=(4, 3),
            textcoords="offset points",
            fontsize=9
        )

    plt.title("Projection PCA en 2D des embeddings Word2Vec")
    plt.xlabel("Composante principale 1")
    plt.ylabel("Composante principale 2")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


plot_word_embeddings(word2vec_model)

### Analyse de la visualisation

Certains mots peuvent apparaître proches, mais cette proximité ne signifie pas toujours qu’ils sont réellement liés sur le plan sémantique.

Les principales raisons sont les suivantes :

1. **Le corpus est très petit.** Il contient seulement dix avis, donc Word2Vec dispose de peu de contextes pour apprendre.
2. **De nombreux mots n’apparaissent qu’une fois.** Le modèle ne peut pas identifier de manière fiable leurs usages habituels.
3. **Les paramètres influencent fortement le résultat.** La taille de la fenêtre, le nombre d’époques et le choix entre Skip-gram et CBOW modifient les positions.
4. **La réduction PCA entraîne une perte d’information.** Les vecteurs de 50 dimensions sont compressés en seulement deux dimensions.
5. **Les résultats sont surtout démonstratifs.** Un modèle Word2Vec performant nécessite normalement un corpus beaucoup plus volumineux.

Ainsi, quelques regroupements peuvent être cohérents, mais il ne faut pas surinterpréter le graphique obtenu avec ce petit échantillon.

## 3. Expérimentation avec différents paramètres

Pour approfondir l’analyse, nous comparons :

- un modèle **Skip-gram** (`sg=1`) ;
- un modèle **CBOW** (`sg=0`) ;
- deux tailles de fenêtre contextuelle différentes.

In [ ]:
# Modèle alternatif utilisant CBOW et une fenêtre plus large

word2vec_cbow = Word2Vec(
    sentences=tokenized_reviews,
    vector_size=50,
    window=5,
    min_count=1,
    workers=1,
    sg=0,
    epochs=300,
    seed=42
)

print("Modèle Skip-gram :", word2vec_model.wv.vectors.shape)
print("Modèle CBOW :", word2vec_cbow.wv.vectors.shape)

comparison_word = "restaurant"

if comparison_word in word2vec_model.wv and comparison_word in word2vec_cbow.wv:
    print(f"\nMots proches de '{comparison_word}' avec Skip-gram :")
    print(word2vec_model.wv.most_similar(comparison_word, topn=5))

    print(f"\nMots proches de '{comparison_word}' avec CBOW :")
    print(word2vec_cbow.wv.most_similar(comparison_word, topn=5))

### Analyse de l’expérimentation

Les voisins d’un même mot peuvent changer lorsque les paramètres du modèle sont modifiés.

- **Skip-gram** essaie de prédire les mots du contexte à partir du mot central. Il peut être intéressant pour les mots rares.
- **CBOW** prédit le mot central à partir de son contexte. Il est souvent rapide et plus stable sur de grands corpus.
- Une **fenêtre plus petite** privilégie les relations locales.
- Une **fenêtre plus grande** capture un contexte plus large, mais peut également introduire du bruit.
- Augmenter le nombre d’époques aide le modèle à parcourir davantage le petit corpus, mais ne remplace pas la diversité d’un grand jeu de données.

Le principal moyen d’améliorer les embeddings serait donc d’utiliser beaucoup plus d’avis, puis d’évaluer plusieurs configurations.

# Conclusion générale

Ce notebook a permis de mettre en pratique plusieurs étapes essentielles d’un pipeline NLP :

- création et conservation de versions brute et nettoyée des données ;
- tokenisation, suppression de la ponctuation et des stopwords, puis lemmatisation ;
- reconnaissance d’entités nommées avec spaCy ;
- analyse grammaticale avec NLTK ;
- entraînement d’un modèle Word2Vec ;
- réduction dimensionnelle et visualisation des embeddings.

La comparaison entre les textes bruts et nettoyés montre qu’un prétraitement agressif n’est pas toujours bénéfique. Il améliore certaines représentations statistiques, mais peut dégrader les tâches qui utilisent la structure linguistique et les noms propres.

Enfin, les embeddings obtenus sont adaptés à une démonstration pédagogique, mais leur qualité reste limitée par la petite taille du corpus.